In [27]:
from ultralytics import YOLO
import cv2
import os
import pandas as pd
import supervision as sv
from pathlib import Path

In [ ]:
yolo_model = {
    "model_1" : "/Users/amritkumar/Desktop/CCTV_VLM_Project/models/yolo11n_E3_PPE_best.pt",
    "model_2" : "/Users/amritkumar/Desktop/CCTV_VLM_Project/models/yolo11s_warehouse_best.pt",
    "model_3" : "/Users/amritkumar/Desktop/CCTV_VLM_Project/models/yolo26s_warehouse_best.pt"
}

project_root = "Users/amritkumar/Desktop/CCTV_VLM_Project"
project_root = Path(project_root)

output_dir = "outputs"
output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

test_videos = [
    "/Users/amritkumar/Desktop/CCTV_VLM_Project/videos/warehouse1.mp4",
    "/Users/amritkumar/Desktop/CCTV_VLM_Project/videos/warehouse2.mp4",
    "/Users/amritkumar/Desktop/CCTV_VLM_Project/videos/warehouse3.mp4",
    "/Users/amritkumar/Desktop/CCTV_VLM_Project/videos/warehouse4.mp4"
]

In [29]:
for name , path in yolo_model.items():
    print(f"Model [{name}] : {"Found" if os.path.exists(path) else "Not Found"}")

for video in test_videos:
    print(f"Video [{video}] : {"Found" if os.path.exists(video) else "Not Found"}")

Model [model_1] : Found
Model [model_2] : Found
Model [model_3] : Found
Video [/Users/amritkumar/Desktop/CCTV_VLM_Project/videos/warehouse1.mp4] : Found
Video [/Users/amritkumar/Desktop/CCTV_VLM_Project/videos/warehouse2.mp4] : Found
Video [/Users/amritkumar/Desktop/CCTV_VLM_Project/videos/warehouse3.mp4] : Found
Video [/Users/amritkumar/Desktop/CCTV_VLM_Project/videos/warehouse4.mp4] : Found


In [30]:
# ==================== CELL 3: TARGETED SHOOTOUT ====================
# Core classes that actually trigger tracking, near-miss, and VLM in our architecture
CORE_TRIGGERS = ["person", "forklift", "pallet_truck"]
PPE_CLASSES = ["helmet", "safety_vest", "safety_shoes"]

def run_targeted_shootout(models_dict, video_paths, conf_thresh=0.25, imgsz=800):
    summary_logs = []
    box_annotator = sv.BoxAnnotator(thickness=2)
    label_annotator = sv.LabelAnnotator(text_scale=0.5, text_thickness=1)

    for model_name, model_path in models_dict.items():
        if not os.path.exists(model_path):
            continue
            
        model = YOLO(model_path)
        class_names = model.names

        for video_path in video_paths:
            if not os.path.exists(video_path):
                continue
                
            video_name = Path(video_path).stem
            out_video_path = str(output_dir / f"{video_name}_{model_name}.mp4")
            video_info = sv.VideoInfo.from_video_path(video_path)
            
            # Track total detections and frame coverage (how many frames had AT LEAST 1 vehicle/person)
            counts = {cls_name: 0 for cls_name in CORE_TRIGGERS + PPE_CLASSES}
            frames_with_vehicle = 0
            frames_with_person = 0
            total_frames = 0

            print(f"🎥 Running {model_name} on {video_name}...")
            
            with sv.VideoSink(target_path=out_video_path, video_info=video_info) as sink:
                for frame in sv.get_video_frames_generator(source_path=video_path):
                    total_frames += 1
                    
                    results = model(frame, conf=conf_thresh, imgsz=imgsz, verbose=False)[0]
                    detections = sv.Detections.from_ultralytics(results)

                    detected_names_in_frame = set()
                    for cls_id in detections.class_id:
                        cls_label = class_names[int(cls_id)]
                        detected_names_in_frame.add(cls_label)
                        if cls_label in counts:
                            counts[cls_label] += 1

                    # Check frame continuity for vehicles & people
                    if "forklift" in detected_names_in_frame or "pallet_truck" in detected_names_in_frame:
                        frames_with_vehicle += 1
                    if "person" in detected_names_in_frame:
                        frames_with_person += 1

                    labels = [
                        f"{class_names[int(cls_id)]} {conf:.2f}"
                        for cls_id, conf in zip(detections.class_id, detections.confidence)
                    ]

                    annotated = box_annotator.annotate(scene=frame.copy(), detections=detections)
                    annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)
                    sink.write_frame(annotated)

            summary_logs.append({
                "Video": video_name,
                "Model": model_name,
                "Frames": total_frames,
                "Vehicle_Frames_%": round((frames_with_vehicle / max(1, total_frames)) * 100, 1),
                "Person_Frames_%": round((frames_with_person / max(1, total_frames)) * 100, 1),
                **counts,
            })
            print(f"   -> Saved: {out_video_path}")

    return pd.DataFrame(summary_logs)

In [33]:
df_shootout = run_targeted_shootout(yolo_model,test_videos ,conf_thresh=0.25, imgsz=800)
display(df_shootout)

🎥 Running model_1 on warehouse1...
   -> Saved: outputs/warehouse1_model_1.mp4
🎥 Running model_1 on warehouse2...
   -> Saved: outputs/warehouse2_model_1.mp4
🎥 Running model_1 on warehouse3...
   -> Saved: outputs/warehouse3_model_1.mp4
🎥 Running model_1 on warehouse4...
   -> Saved: outputs/warehouse4_model_1.mp4
🎥 Running model_2 on warehouse1...
   -> Saved: outputs/warehouse1_model_2.mp4
🎥 Running model_2 on warehouse2...
   -> Saved: outputs/warehouse2_model_2.mp4
🎥 Running model_2 on warehouse3...
   -> Saved: outputs/warehouse3_model_2.mp4
🎥 Running model_2 on warehouse4...
   -> Saved: outputs/warehouse4_model_2.mp4
🎥 Running model_3 on warehouse1...
   -> Saved: outputs/warehouse1_model_3.mp4
🎥 Running model_3 on warehouse2...
   -> Saved: outputs/warehouse2_model_3.mp4
🎥 Running model_3 on warehouse3...
   -> Saved: outputs/warehouse3_model_3.mp4
🎥 Running model_3 on warehouse4...
   -> Saved: outputs/warehouse4_model_3.mp4


,Video,Model,Frames,Vehicle_Frames_%,Person_Frames_%,person,forklift,pallet_truck,helmet,safety_vest,safety_shoes
0,warehouse1,model_1,198,0.0,83.8,182,0,0,0,0,0
1,warehouse2,model_1,173,0.0,95.4,303,0,0,0,0,0
2,warehouse3,model_1,198,0.0,96.0,296,0,0,0,0,0
3,warehouse4,model_1,173,1.2,91.3,291,2,0,0,0,0
4,warehouse1,model_2,198,0.0,59.6,231,0,0,0,0,0
5,warehouse2,model_2,173,0.0,100.0,321,0,0,0,0,0
6,warehouse3,model_2,198,0.0,82.3,281,0,0,0,0,0
7,warehouse4,model_2,173,0.0,100.0,572,0,0,44,0,24
8,warehouse1,model_3,198,0.0,0.0,0,0,0,0,0,0
9,warehouse2,model_3,173,20.8,91.3,244,36,0,0,0,0
